# VideoSense — Experiments (Steps 5–12)

This notebook runs every training and evaluation command in order. The code lives in `MyDrive/videosense/code/` (the project folder). This notebook only calls the scripts. **Read `EXPERIMENT_GUIDE.md` first**: it explains every hyperparameter used below.

| Step | Assignment | Runs |
|---|---|---|
| 5 | Part 2, Ex 6–8 | C3D baseline + label-smoothing sweep |
| 6 | Part 2, Ex 9–10 | pretrained R(2+1)D, complexity |
| 7 | Part 3, Ex 14–16 | Siamese (contrastive) + margin sweep |
| 8 | Part 3, Ex 17 | triplet vs contrastive |
| 9 | Part 4, Ex 21–22 | RGB / flow / fusion ablation |
| 10 | Part 4, Ex 23–24 | attention fusion, per-class analysis |
| 11 | Part 5, Ex 28–32 | report, confusion, t-SNE, Grad-CAM, TTA |
| 12 | Section 11 | full-dataset runs, export, packaging |

> **Runtime: GPU** (`Runtime → Change runtime type → T4 GPU`). Every run saves to Drive after each epoch. If Colab disconnects, re-run the same cell with `--resume` added.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L

import os, json, glob, subprocess
ROOT     = '/content/drive/MyDrive/videosense'
CODE_DIR = f'{ROOT}/code'
META     = f'{ROOT}/metadata'
RUNS     = f'{ROOT}/runs'
FIGS     = f'{ROOT}/figures/results'
DATA     = '/content/videosense/data'            # fast local disk (rebuilt every session)
SUBSET   = 'mini'                                # switch to 'full' in Step 12
os.makedirs(RUNS, exist_ok=True); os.makedirs(DATA, exist_ok=True)
assert os.path.exists(f'{CODE_DIR}/train_3dcnn.py'), 'Upload the project folder to MyDrive/videosense/code first'
assert os.path.exists(f'{META}/preprocess_config.json'), 'Run notebook 01 (Steps 1-4) first'
cfg = json.load(open(f'{META}/preprocess_config.json'))
print(json.dumps(cfg, indent=1))

In [ ]:
!pip -q install torchinfo imageio
%cd {CODE_DIR}

### Restore the caches to local disk

Each new GPU session starts with an empty local disk. The caches are restored from the tar backups made in notebook 01. That takes a few minutes for the mini subset, longer for the full one.

In [ ]:
def restore(name):
    target = f'{DATA}/{name}'
    tar = f'{ROOT}/cache/{name}_{SUBSET}.tar'
    if os.path.isdir(target) and len(glob.glob(f'{target}/*/*.npz')) > 0:
        print(f'{name}: already on local disk'); return
    assert os.path.exists(tar), f'{tar} missing: build it in notebook 01 with SUBSET="{SUBSET}"'
    print(f'{name}: extracting {os.path.getsize(tar)/1e9:.2f} GB ...')
    subprocess.run(['tar', '-xf', tar, '-C', DATA], check=True)

restore(cfg['CACHE_NAME'])
restore(cfg['FLOW_NAME'])
for name in (cfg['CACHE_NAME'], cfg['FLOW_NAME']):
    print(name, len(glob.glob(f'{DATA}/{name}/*/*.npz')), 'clips')

In [ ]:
# Arguments shared by every command below
COMMON = f'--data-root {DATA} --meta-dir {META} --runs-dir {RUNS} --subset {SUBSET} --num-workers 2'
EVAL   = f'--data-root {DATA} --meta-dir {META} --runs-dir {RUNS} --out-dir {FIGS}'
print(COMMON)

### Smoke test (about 5 minutes; do this before any long run)

Runs every script for one tiny epoch. Path, cache, or memory errors show up here, not an hour into a run. The smoke runs are deleted afterwards.

In [ ]:
SMOKE = f'{COMMON} --epochs 1 --max-batches 3 --batch-size 4'
!python train_3dcnn.py   {SMOKE} --run-name smoke_c3d
!python train_siamese.py {SMOKE} --run-name smoke_siam --pairs-per-epoch 24 --val-pairs 24 --episodes 10
!python train_fusion.py  {SMOKE} --run-name smoke_fus --mode attention
!python evaluate.py tsne {EVAL} --runs {RUNS}/smoke_c3d {RUNS}/smoke_fus --max-batches 3

In [ ]:
import shutil
for r in glob.glob(f'{RUNS}/smoke_*'):
    shutil.rmtree(r)
print('smoke runs removed')

## Step 5 — C3D baseline (Exercises 6–8)

`evaluate()` (Exercise 6) is `utils/common.py::evaluate` (top-1/top-5). Training uses validation (groups 8–10) to pick the best epoch and runs the test set once at the end.

**What to look for:**
- **Epoch-1 loss** should start near ln(10) ≈ 2.3 and fall. If it's in the hundreds, the learning rate is too high (EXPERIMENT_GUIDE §3.1).
- **Train vs val curves:** a widening gap means overfitting.
- After the first epoch, read `time_s` in the printed line and multiply by 30 to plan your session.

In [ ]:
!python train_3dcnn.py {COMMON} --run-name c3d_ls0.1 --model c3d --label-smoothing 0.1 --epochs 30

In [ ]:
# Exercise 8: label-smoothing sweep (same everything else)
!python train_3dcnn.py {COMMON} --run-name c3d_ls0.0 --model c3d --label-smoothing 0.0 --epochs 30
!python train_3dcnn.py {COMMON} --run-name c3d_ls0.2 --model c3d --label-smoothing 0.2 --epochs 30

In [ ]:
# Exercise 7 + 8 figures
!python evaluate.py curves {EVAL} --runs {RUNS}/c3d_ls0.0 {RUNS}/c3d_ls0.1 {RUNS}/c3d_ls0.2 --labels "LS 0.0" "LS 0.1" "LS 0.2" --out {FIGS}/part2_c3d_label_smoothing.png --title "C3D: label smoothing"
!python evaluate.py summary {EVAL}

**Decide:** keep the smoothing value with the best `best_val_top1`; within ~1 point, prefer the smaller train–val gap. Use this run (`c3d_ls<best>`) as "the C3D" in later steps.

*Optional, and a good Methods-section point:* run `--lr 1e-3` (the assignment's value) to show the first-step loss spike.

In [ ]:
BEST_C3D = f'{RUNS}/c3d_ls0.1'      # <- change to your best label-smoothing run

## Step 6 — Pretrained R(2+1)D-18 (Exercises 9–10, Q12)

`--finetune head` trains only the new classifier; the Kinetics backbone is frozen, with its BatchNorm statistics kept as well. Optionally, `layer4` also fine-tunes the last stage with a 10× smaller learning rate.

In [ ]:
!python train_3dcnn.py {COMMON} --run-name r2p1d_head --model r2plus1d --finetune head --epochs 10 --lr 1e-3 --batch-size 8

In [ ]:
# Optional: partial fine-tuning (usually the best accuracy for the compute)
!python train_3dcnn.py {COMMON} --run-name r2p1d_layer4 --model r2plus1d --finetune layer4 --epochs 10 --lr 1e-3 --backbone-lr-mult 0.1

In [ ]:
# Exercise 10: parameters, multiply-adds (FLOPs ~ 2x), latency
!python evaluate.py complexity {EVAL} --num-classes 101
!python evaluate.py curves {EVAL} --runs {BEST_C3D} {RUNS}/r2p1d_head --labels "C3D scratch" "R(2+1)D head" --out {FIGS}/part2_c3d_vs_r2p1d.png
!python evaluate.py summary {EVAL}

## Step 7 — Siamese network (Exercises 14–16, Q18–Q20)

Classes are split into **5 base classes** (training) and **5 novel classes** (few-shot evaluation only). The encoder is initialized from a C3D trained on the **base classes only**, so the novel classes are never seen, not even through the initialization (EXPERIMENT_GUIDE §4).

In [ ]:
!python train_3dcnn.py {COMMON} --run-name c3d_base --model c3d --exclude-novel 5 --label-smoothing 0.1 --epochs 30

In [ ]:
# Exercises 14-16: margin sweep (distances lie in [0, 2] on the unit sphere)
for m in ['1.0', '2.0', '4.0']:
    !python train_siamese.py {COMMON} --run-name siam_m{m} --loss contrastive --margin {m} --init-from {RUNS}/c3d_base --epochs 20

In [ ]:
!python evaluate.py distances {EVAL} --runs {RUNS}/siam_m1.0 {RUNS}/siam_m2.0 {RUNS}/siam_m4.0 --labels "margin 1" "margin 2" "margin 4"
!python evaluate.py summary {EVAL}

**What to look for:** with margin 1, the positive and negative histograms separate and the negatives settle near or above 1. With margin 4 (unreachable), the negatives keep pushing, so watch whether the positive distances stop shrinking.

*Optional Q19 experiment:* 90% negative pairs, i.e. `--pos-fraction 0.1` (the argument is the share of **positive** pairs).

In [ ]:
!python train_siamese.py {COMMON} --run-name siam_neg90 --loss contrastive --margin 1.0 --pos-fraction 0.1 --init-from {RUNS}/c3d_base --epochs 20

## Step 8 — Triplet vs contrastive (Exercise 17)

All modes encode the same number of clips per epoch, so the comparison is at equal compute.

In [ ]:
!python train_siamese.py {COMMON} --run-name siam_triplet --loss triplet --margin 0.3 --init-from {RUNS}/c3d_base --epochs 20
!python train_siamese.py {COMMON} --run-name siam_bh --loss triplet_batchhard --margin 0.3 --pk-p 5 --pk-k 4 --init-from {RUNS}/c3d_base --epochs 20
!python evaluate.py distances {EVAL} --runs {RUNS}/siam_m1.0 {RUNS}/siam_triplet {RUNS}/siam_bh --labels contrastive triplet batch-hard
!python evaluate.py summary {EVAL}

In [ ]:
BEST_SIAM = f'{RUNS}/siam_m1.0'     # <- your best Siamese run by best_val_auc

## Step 9 — Two-stream ablation (Exercises 21–22)

All three runs read identical (RGB, flow) batches; only the model differs.

In [ ]:
!python train_fusion.py {COMMON} --run-name fus_rgb    --mode rgb    --epochs 30
!python train_fusion.py {COMMON} --run-name fus_flow   --mode flow   --epochs 30
!python train_fusion.py {COMMON} --run-name fus_concat --mode concat --epochs 30

In [ ]:
!python evaluate.py curves {EVAL} --runs {RUNS}/fus_rgb {RUNS}/fus_flow {RUNS}/fus_concat --labels RGB Flow "RGB+Flow" --out {FIGS}/part4_ablation_curves.png
!python evaluate.py summary {EVAL}

## Step 10 — Attention fusion and per-class analysis (Exercises 23–24, Q27)

`attention` learns one scalar α (the RGB weight), with its own 10× learning rate and no weight decay, so it can actually move away from its 0.5 starting value. The stronger variant initializes each stream from its single-stream run (EXPERIMENT_GUIDE §3.4).

In [ ]:
!python train_fusion.py {COMMON} --run-name fus_attn      --mode attention --epochs 30
!python train_fusion.py {COMMON} --run-name fus_attn_init --mode attention --epochs 15 --init-rgb {RUNS}/fus_rgb --init-flow {RUNS}/fus_flow --backbone-lr-mult 0.1

In [ ]:
# Optional: other fusion strategies from the Section 8.4 table
!python train_fusion.py {COMMON} --run-name fus_avg   --mode avg   --epochs 30
!python train_fusion.py {COMMON} --run-name fus_gated --mode gated --epochs 15 --init-rgb {RUNS}/fus_rgb --init-flow {RUNS}/fus_flow --backbone-lr-mult 0.1

In [ ]:
import pandas as pd
h = pd.read_csv(f'{RUNS}/fus_attn/history.csv')
print(h[['epoch', 'val_top1', 'alpha_rgb']].to_string(index=False))

# Exercise 24: classes where flow helps most (fusion vs RGB-only)
!python evaluate.py compare {EVAL} --run-a {RUNS}/fus_rgb --run-b {RUNS}/fus_concat --labels "RGB only" "RGB+Flow"
!python evaluate.py summary {EVAL}

In [ ]:
BEST_FUSION = f'{RUNS}/fus_concat'  # <- your best fusion run by best_val_top1

## Step 11 — Evaluation and interpretation (Exercises 28–32, Q33–Q35)

In [ ]:
# Exercises 28-29: full report + confusion matrix + most-confused pairs
!python evaluate.py report {EVAL} --run {BEST_FUSION}
!python evaluate.py report {EVAL} --run {BEST_C3D}

In [ ]:
# Exercise 30: t-SNE + silhouette for the three model families
!python evaluate.py tsne {EVAL} --runs {BEST_C3D} {BEST_SIAM} {BEST_FUSION} --labels C3D Siamese "Two-stream"

In [ ]:
# Exercise 31: Grad-CAM GIFs (+ a frame-strip PNG for the PDF report)
!python evaluate.py gradcam {EVAL} --run {BEST_C3D} --n-clips 5
!python evaluate.py gradcam {EVAL} --run {BEST_FUSION} --n-clips 5

In [ ]:
# Exercise 32 (bonus): test-time augmentation with 5 temporal crops
!python evaluate.py tta {EVAL} --run {BEST_C3D} --n-crops 5
!python evaluate.py tta {EVAL} --run {BEST_FUSION} --n-crops 5

In [ ]:
from IPython.display import Image, display
for f in sorted(glob.glob(f'{FIGS}/*.png')):
    print(os.path.basename(f)); display(Image(f, width=900))

## Step 12 — Final runs, export, and packaging (Section 11)

1. **Full dataset (if compute allows):** build the full caches in notebook 01 (`SUBSET = 'full'`, cells 3.3 and 4.5), set `SUBSET = 'full'` in the setup cell above, re-run the setup cells, then repeat only your **chosen** configurations. Estimate the time from epoch 1 before committing to a long run.
2. **Export** the three required checkpoints as plain `state_dict` files.
3. **Package** the code, checkpoints, and requirements.

In [ ]:
# Export the required checkpoints (Section 11.2) from your chosen runs
from utils.common import export_state_dict
os.makedirs(f'{CODE_DIR}/models', exist_ok=True)
export_state_dict(BEST_C3D,    f'{CODE_DIR}/models/c3d_ucf101.pt')
export_state_dict(BEST_SIAM,   f'{CODE_DIR}/models/siamese_ucf101.pt')
export_state_dict(BEST_FUSION, f'{CODE_DIR}/models/twostream_ucf101.pt')

In [ ]:
# Exact environment + style check (Section 11.1)
!pip freeze > {CODE_DIR}/requirements_frozen.txt
!pip -q install flake8 && cd {CODE_DIR} && flake8 --max-line-length 120 --exclude notebooks . && echo "flake8: clean"

In [ ]:
# Final results table for the report
!python evaluate.py summary {EVAL}

In [ ]:
LAST, FIRST = 'Kaur', 'Rachandeep'        # <- check the spelling you want on the file name
ZIP = f'{ROOT}/{LAST}_{FIRST}_VideoSense.zip'
!cd {ROOT} && rm -f {ZIP} && zip -qr {ZIP} code -x "code/**/__pycache__/*" && ls -lh {ZIP}

**Before submitting:** check that the zip contains the three `.pt` files and `README.md`. Submit the PDF report separately or inside the zip, as your instructor asks. Remember the report needs the reflection answers. `EXPERIMENT_GUIDE.md` §5 lists where the evidence for each question is.